In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import torch
from transformers import AutoModel, AutoTokenizer, BartForConditionalGeneration
from sklearn.metrics.pairwise import cosine_similarity
from nltk.tokenize import sent_tokenize
import numpy as np

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load models
bert_model = AutoModel.from_pretrained("law-ai/InLegalBERT").to(device)
bert_tokenizer = AutoTokenizer.from_pretrained("law-ai/InLegalBERT")

bart_model = BartForConditionalGeneration.from_pretrained("/content/drive/MyDrive/fine_tuned_bart_legal").to(device)
bart_tokenizer = AutoTokenizer.from_pretrained("/content/drive/MyDrive/fine_tuned_bart_legal")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/671 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/534M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/534M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/516 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/222k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [5]:
# Helper to get sentence embeddings
def get_embeddings_batch(sentences):
    inputs = bert_tokenizer(sentences, padding=True, truncation=True, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = bert_model(**inputs)
    embeddings = outputs.last_hidden_state.mean(dim=1)
    return embeddings.cpu().numpy()

In [6]:
# Extractive Summary
def extractive_summary(chunk, summary_length):
    embeddings = get_embeddings_batch(chunk)
    print("\n[Extractive] Embeddings shape:", embeddings.shape)

    sim_matrix = cosine_similarity(embeddings)
    print("[Extractive] Cosine similarity matrix:\n", sim_matrix)

    scores = sim_matrix.sum(axis=1)
    print("[Extractive] Sentence scores:", scores)

    top_indices = np.argsort(scores)[-summary_length:][::-1]
    print("[Extractive] Top sentence indices:", top_indices)

    top_sents = [chunk[i] for i in sorted(top_indices)]
    return " ".join(top_sents)

In [7]:
# Abstractive Summary
def abstractive_summary(extractive_summary, chunk_summary_max, chunk_summary_min):
    inputs = bart_tokenizer(extractive_summary, return_tensors="pt", truncation=True, max_length=1024).to(device)
    summary_ids = bart_model.generate(
        **inputs,
        max_length=chunk_summary_max,
        min_length=chunk_summary_min,
        num_beams=4,
        length_penalty=2.0,
        no_repeat_ngram_size=3,
        repetition_penalty=1.2,
        early_stopping=True,
    )
    decoded = bart_tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    print("[Abstractive] Generated summary:\n", decoded)
    return decoded

In [8]:
# Hybrid Summary
def hybrid_summary(chunk, chunk_summary_max, chunk_summary_min):
    extractive = extractive_summary(chunk, summary_length=2)
    print("[Hybrid] Extractive step output:\n", extractive)
    return abstractive_summary(extractive, chunk_summary_max, chunk_summary_min)

Sample Text :

In the present matter, the accused Samit was initially charged under Section 420 of the Indian Penal Code (IPC) for the alleged act of cheating one Jaden of an amount totaling ₹50,00,000. The charge was based on a complaint filed by Jaden, asserting that Samit had fraudulently acquired the said sum.

However, during the course of investigation, crucial details emerged that shifted the narrative. It was found that during a private gathering on 30th May 2025, held at a deserted island, a third individual, identified as Ralph P., played a central role in deceiving Samit and taking possession of the ₹50 lakhs.

Subsequent findings revealed that the amount in question had originally been unlawfully acquired by Samit from Jaden. Nevertheless, the investigating authorities could not establish any direct or circumstantial evidence conclusively linking Samit to the act of intentional cheating under Section 420 IPC.

In light of the above, the Hon’ble Court observed that the charges framed by the Sessions Court lacked substantive evidence and thereby ordered that the case be quashed. The bench stated unequivocally that, in the absence of clear proof indicating fraudulent intent, the benefit of doubt must lie with the accused.

In [39]:
# === Run a sample input ===
text = """
In the present matter, the accused Samit was initially charged under Section 420 of the Indian Penal Code (IPC) for the alleged act of cheating one Jaden of an amount totaling ₹50,00,000. The charge was based on a complaint filed by Jaden, asserting that Samit had fraudulently acquired the said sum.

However, during the course of investigation, crucial details emerged that shifted the narrative. It was found that during a private gathering on 30th May 2025, held at a deserted island, a third individual, identified as Ralph P., played a central role in deceiving Samit and taking possession of the ₹50 lakhs.

Subsequent findings revealed that the amount in question had originally been unlawfully acquired by Samit from Jaden. Nevertheless, the investigating authorities could not establish any direct or circumstantial evidence conclusively linking Samit to the act of intentional cheating under Section 420 IPC.

In light of the above, the Hon’ble Court observed that the charges framed by the Sessions Court lacked substantive evidence and thereby ordered that the case be quashed. The bench stated unequivocally that, in the absence of clear proof indicating fraudulent intent, the benefit of doubt must lie with the accused.
"""

sentences = preprocess_text_to_sentences(text)

In [38]:
import nltk

nltk.download('punkt_tab')
nltk.download("punkt")

from nltk.tokenize import sent_tokenize

def preprocess_text_to_sentences(text):
    return sent_tokenize(text)


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


In [40]:
sentences = sent_tokenize(text)
chunk = sentences[:7]  # simulate one chunk

print("📝 Original Chunk:\n", "\n".join(chunk))

📝 Original Chunk:
 
In the present matter, the accused Samit was initially charged under Section 420 of the Indian Penal Code (IPC) for the alleged act of cheating one Jaden of an amount totaling ₹50,00,000.
The charge was based on a complaint filed by Jaden, asserting that Samit had fraudulently acquired the said sum.
However, during the course of investigation, crucial details emerged that shifted the narrative.
It was found that during a private gathering on 30th May 2025, held at a deserted island, a third individual, identified as Ralph P., played a central role in deceiving Samit and taking possession of the ₹50 lakhs.
Subsequent findings revealed that the amount in question had originally been unlawfully acquired by Samit from Jaden.
Nevertheless, the investigating authorities could not establish any direct or circumstantial evidence conclusively linking Samit to the act of intentional cheating under Section 420 IPC.
In light of the above, the Hon’ble Court observed that the cha

In [41]:
print("\n=== Extractive Summary ===")
ext_sum = extractive_summary(sentences, summary_length=3)
print(ext_sum)


=== Extractive Summary ===

[Extractive] Embeddings shape: (8, 768)
[Extractive] Cosine similarity matrix:
 [[1.0000002  0.81739974 0.72608006 0.840272   0.77919227 0.8311473
  0.794039   0.7884659 ]
 [0.81739974 1.0000002  0.8451571  0.8138668  0.9036335  0.8433831
  0.7816274  0.8616726 ]
 [0.72608006 0.8451571  1.0000001  0.7526643  0.84694344 0.81386435
  0.75180614 0.8236992 ]
 [0.840272   0.8138668  0.7526643  0.99999994 0.792726   0.8179245
  0.78286505 0.7743028 ]
 [0.77919227 0.9036335  0.84694344 0.792726   1.0000002  0.83686906
  0.7838762  0.8448466 ]
 [0.8311473  0.8433831  0.81386435 0.8179245  0.83686906 1.
  0.793443   0.8572875 ]
 [0.794039   0.7816274  0.75180614 0.78286505 0.7838762  0.793443
  1.0000004  0.83131266]
 [0.7884659  0.8616726  0.8236992  0.7743028  0.8448466  0.8572875
  0.83131266 1.0000002 ]]
[Extractive] Sentence scores: [6.5765967 6.86674   6.560215  6.574621  6.788087  6.7939186 6.51897
 6.7815876]
[Extractive] Top sentence indices: [1 5 4]
The ch

The charge was based on a complaint filed by Jaden, asserting that Samit had fraudulently acquired the said sum. Subsequent findings revealed that the amount in question had originally been unlawfully acquired by Samit from Jaden. Nevertheless, the investigating authorities could not establish any direct or circumstantial evidence conclusively linking Samit to the act of intentional cheating under Section 420 IPC.



In [47]:
print("\n=== Abstractive Summary ===")
abs_sum = abstractive_summary(text, chunk_summary_max=150, chunk_summary_min=50)
print(abs_sum)


=== Abstractive Summary ===
[Abstractive] Generated summary:
 The accused Samit was initially charged under Section 420 of the Indian Penal Code (IPC) for the alleged act of cheating one Jaden of an amount totaling ₹50 lakhs. The charge was based on a complaint filed by Jaden, asserting that Samit had fraudulently acquired the said sum. During the course of investigation, crucial details emerged that shifted the narrative. It was found that during a private gathering on 30th May 2025 held at a deserted island, a third individual, identified as Ralph P., played a central role in deceiving Samit and taking possession of the said amount. Subsequent findings revealed that the amount in question had originally been unlawfully acquired by Samit from Jaden. Nevertheless, the
The accused Samit was initially charged under Section 420 of the Indian Penal Code (IPC) for the alleged act of cheating one Jaden of an amount totaling ₹50 lakhs. The charge was based on a complaint filed by Jaden, asse

The accused Samit was initially charged under Section 420 of the Indian Penal Code (IPC) for the alleged act of cheating one Jaden of an amount totaling ₹50 lakhs. The charge was based on a complaint filed by Jaden, asserting that Samit had fraudulently acquired the said sum. During the course of investigation, crucial details emerged that shifted the narrative. It was found that during a private gathering on 30th May 2025 held at a deserted island, a third individual, identified as Ralph P., played a central role in deceiving Samit and taking possession of the said amount. Subsequent findings revealed that the amount in question had originally been unlawfully acquired by Samit from Jaden. Nevertheless, the

In [43]:
print("\n=== Hybrid Summary ===")
hyb_sum = hybrid_summary(sentences, chunk_summary_max=150, chunk_summary_min=50)
print(hyb_sum)


=== Hybrid Summary ===

[Extractive] Embeddings shape: (8, 768)
[Extractive] Cosine similarity matrix:
 [[1.0000002  0.81739974 0.72608006 0.840272   0.77919227 0.8311473
  0.794039   0.7884659 ]
 [0.81739974 1.0000002  0.8451571  0.8138668  0.9036335  0.8433831
  0.7816274  0.8616726 ]
 [0.72608006 0.8451571  1.0000001  0.7526643  0.84694344 0.81386435
  0.75180614 0.8236992 ]
 [0.840272   0.8138668  0.7526643  0.99999994 0.792726   0.8179245
  0.78286505 0.7743028 ]
 [0.77919227 0.9036335  0.84694344 0.792726   1.0000002  0.83686906
  0.7838762  0.8448466 ]
 [0.8311473  0.8433831  0.81386435 0.8179245  0.83686906 1.
  0.793443   0.8572875 ]
 [0.794039   0.7816274  0.75180614 0.78286505 0.7838762  0.793443
  1.0000004  0.83131266]
 [0.7884659  0.8616726  0.8236992  0.7743028  0.8448466  0.8572875
  0.83131266 1.0000002 ]]
[Extractive] Sentence scores: [6.5765967 6.86674   6.560215  6.574621  6.788087  6.7939186 6.51897
 6.7815876]
[Extractive] Top sentence indices: [1 5]
[Hybrid] Ext

The charge was based on a complaint filed by Jaden, asserting that Samit had fraudulently acquired the said sum. Nevertheless, the investigating authorities could not establish any direct or circumstantial evidence conclusively linking Samit to the act of intentional cheating under section 420 I.P.C. The charge sheet was accordingly framed against Samit. The accused was produced before the Magistrate who convicted him on the ground that he had committed an offence under Section 420 of the Indian Penal Code and sentenced him to rigorous imprisonment for one year.